# Immunogenicity_Findings — graph-fed consumer

Builds `sdtm-findings-graph/machine_actionable/Immunogenicity_Findings.xlsx` for the
immunogenicity sub-type of Findings. Sheets:

| Sheet | Grain | Source |
|---|---|---|
| `Test_Identity` | TESTCD + NCIt code | `SDTM_Test_Identity.xlsx` filtered to scope; COSMoS coverage flags from `DSS_View.xlsx` |
| `Measurement_Specs` | DSS | `DSS_View.xlsx` filtered to scope; binding agent as its own column block (`BDAGNT_*`) |
| `Axis_Meaning` | `Variable_Meaning` row | `Variable_Meaning` in `SDTM_Domain_Metadata.xlsx`, rows for IS and the axis rows |
| `Codelists` | codelist term | every codelist named in a `*_codelist` column, from `COSMoS_Graph_CT.xlsx` |

## Scope

Immunogenicity Specimen Assessments (SDTMIG v3.4 section 6.3.5.5). The DSSs of one BC differ
by binding agent (`--BDAGNT`), not by specimen, so IS is excluded from `Specimen_Findings`
(`Consumer_Exclusions` IS-1, IS-2) and has this file of its own.

Scope is derived from `SDTM_Domain_Metadata.xlsx` (`Domains` sheet, `Immunogenicity = Yes`)
minus the exclusions in `Consumer_Exclusions` for `Immunogenicity_Findings`.
`Immunogenicity` is checked before `Specimen_Based` wherever a consumer class is assigned
(`Scope_Check`, `Domain_Coverage`, `Domain_Behaviour`).

**In scope:** IS (at SDTM CT 2026-09-25 / COSMoS 2026-07-14). **Excluded:** none.

No observable key here: Observables Phase 2 built the key for LB, MB and MI only.
`Axis_Meaning` says which columns identify the observable.

## Inputs

| File | Track | Sheets used |
|---|---|---|
| `consumer-bases/interim/DSS_View.xlsx` | consumer-bases | `Test_Identity`, `Measurement_Specs` |
| `sdtm-test-codes/machine_actionable/SDTM_Test_Identity.xlsx` | sdtm-test-codes | `Test Codes` |
| `sdtm-domain-reference/machine_actionable/SDTM_Domain_Metadata.xlsx` | sdtm-domain-reference | `Domains`, `Consumer_Exclusions`, `Variable_Meaning` |
| `cosmos-graph/interim/COSMoS_Graph_CT.xlsx` | cosmos-graph | `Codelists`, `CodelistTerms` (Allowed_Units, Codelists sheet) |

## Output

`sdtm-findings-graph/machine_actionable/Immunogenicity_Findings.xlsx`


## 1. Setup

In [1]:
import json
import pandas as pd
from pathlib import Path
from datetime import datetime
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

print('IMMUNOGENICITY_FINDINGS — graph-fed consumer')
print(f'Run: {datetime.now():%Y-%m-%d %H:%M}')

IMMUNOGENICITY_FINDINGS — graph-fed consumer
Run: 2026-10-31 21:22


In [2]:
BASE_DIR = Path.cwd().parent          # sdtm-findings-graph/
REPO_ROOT = BASE_DIR.parent           # cdisc-for-ai/
VERSIONS_FILE = REPO_ROOT / 'Reference_Versions.json'
VERSIONS = json.loads(VERSIONS_FILE.read_text(encoding='utf-8'))

DSS_VIEW_FILE = REPO_ROOT / 'consumer-bases' / 'interim' / 'DSS_View.xlsx'
TEST_IDENTITY_FILE = REPO_ROOT / 'sdtm-test-codes' / 'machine_actionable' / 'SDTM_Test_Identity.xlsx'
DOMAIN_META_FILE = REPO_ROOT / 'sdtm-domain-reference' / 'machine_actionable' / 'SDTM_Domain_Metadata.xlsx'
GRAPH_CT_FILE = REPO_ROOT / 'cosmos-graph' / 'interim' / 'COSMoS_Graph_CT.xlsx'

OUTPUT_DIR = BASE_DIR / 'machine_actionable'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / 'Immunogenicity_Findings.xlsx'

# Immunogenicity-scope domains: SDTM_Domain_Metadata.xlsx Domains sheet, Immunogenicity = Yes,
# minus the exclusions in its Consumer_Exclusions sheet (Consumer = Immunogenicity_Findings).
# Immunogenicity is checked before Specimen_Based: IS is Specimen_Based = Yes as well (IG 6.3.5).

# Allowed_Units expansion: expand ORRESU_codelist into permissible-value list only
# when the codelist has at most this many terms. UNIT (~950 terms) is too broad for
# per-row expansion; narrower sub-codelists are useful. Same threshold as Measurement_Findings.
ALLOWED_UNITS_THRESHOLD = 50

for f, label in [
    (DSS_VIEW_FILE, 'DSS_View'),
    (TEST_IDENTITY_FILE, 'SDTM_Test_Identity'),
    (DOMAIN_META_FILE, 'SDTM_Domain_Metadata'),
    (GRAPH_CT_FILE, 'COSMoS_Graph_CT'),
]:
    if not f.exists():
        raise FileNotFoundError(f'{label} not found: {f}')
    print(f'  {label}: {f.relative_to(REPO_ROOT)}')

_meta = pd.read_excel(DOMAIN_META_FILE, sheet_name='Domains', dtype=str)
_excl = pd.read_excel(DOMAIN_META_FILE, sheet_name='Consumer_Exclusions', dtype=str).fillna('')
_specimen_excl = _excl[_excl['Consumer'] == 'Specimen_Findings']
_excl = _excl[_excl['Consumer'] == 'Immunogenicity_Findings']
EXCLUDED_DOMAINS = list(dict.fromkeys(_excl['Domain']))
EXCLUSION_REASONS = dict(zip(_excl['Domain'], _excl['Reason']))
_immuno = _meta.loc[_meta['Immunogenicity'] == 'Yes', 'Domain'].tolist()
_unknown = sorted(set(EXCLUDED_DOMAINS) - set(_immuno))
if _unknown:
    raise ValueError(f'EXCLUDED_DOMAINS not flagged Immunogenicity = Yes in SDTM_Domain_Metadata: {_unknown}')
SCOPE_DOMAINS = [d for d in _immuno if d not in EXCLUDED_DOMAINS]
# Why these domains are not in Specimen_Findings: its Consumer_Exclusions rows, stated in the ReadMe.
SPECIMEN_EXCLUSION = {d: (', '.join(g['Claim_ID']), '; '.join(dict.fromkeys(g['Reason'])))
                      for d, g in _specimen_excl.groupby('Domain') if d in SCOPE_DOMAINS}

print(f'  Output: {OUTPUT_FILE.relative_to(REPO_ROOT)}')
print(f'  Scope:  {SCOPE_DOMAINS}')
print(f'  Excluded: {EXCLUDED_DOMAINS}')
for _d, _r in EXCLUSION_REASONS.items():
    print(f'    {_d}: {_r}')
print(f'  Allowed_Units threshold: {ALLOWED_UNITS_THRESHOLD} terms')

  DSS_View: consumer-bases/interim/DSS_View.xlsx
  SDTM_Test_Identity: sdtm-test-codes/machine_actionable/SDTM_Test_Identity.xlsx
  SDTM_Domain_Metadata: sdtm-domain-reference/machine_actionable/SDTM_Domain_Metadata.xlsx
  COSMoS_Graph_CT: cosmos-graph/interim/COSMoS_Graph_CT.xlsx
  Output: sdtm-findings-graph/machine_actionable/Immunogenicity_Findings.xlsx
  Scope:  ['IS']
  Excluded: []
  Allowed_Units threshold: 50 terms


## 2. Load inputs

In [3]:
view_ti = pd.read_excel(DSS_VIEW_FILE, sheet_name='Test_Identity', dtype=str).fillna('')
view_ms = pd.read_excel(DSS_VIEW_FILE, sheet_name='Measurement_Specs', dtype=str).fillna('')
_cosmos_newest = view_ms['package_date'].max()
if _cosmos_newest != VERSIONS['COSMoS']:
    raise RuntimeError(f"COSMoS pin {VERSIONS['COSMoS']} != newest package_date {_cosmos_newest} in DSS_View: update Reference_Versions.json")
ref_tests = pd.read_excel(TEST_IDENTITY_FILE, sheet_name='Test Codes', dtype=str).fillna('')
domain_meta = pd.read_excel(DOMAIN_META_FILE, sheet_name='Domains', dtype=str).fillna('')
ct_terms = pd.read_excel(GRAPH_CT_FILE, sheet_name='CodelistTerms', dtype=str, keep_default_na=False)

print(f'DSS_View Test_Identity:     {len(view_ti):>6,} rows  (COSMoS-pinned TESTCDs, graph-wide)')
print(f'DSS_View Measurement_Specs: {len(view_ms):>6,} rows  (DSS-grain, all 32 domains)')
print(f'SDTM_Test_Identity:         {len(ref_tests):>6,} rows  (TESTCDs scoped via SDTM_Domains)')
print(f'SDTM_Domain_Metadata:       {len(domain_meta):>6,} rows')
print(f'COSMoS CodelistTerms:       {len(ct_terms):>6,} rows')

DSS_View Test_Identity:        829 rows  (COSMoS-pinned TESTCDs, graph-wide)
DSS_View Measurement_Specs:  1,475 rows  (DSS-grain, all 32 domains)
SDTM_Test_Identity:          5,988 rows  (TESTCDs scoped via SDTM_Domains)
SDTM_Domain_Metadata:           66 rows
COSMoS CodelistTerms:       17,742 rows


## 3. Build Test_Identity

Universe = `SDTM_Test_Identity` rows where `SDTM_Domains` intersects the
immunogenicity-scope domain set. The COSMoS-pinned subset is a flag (`Has_DSS`),
not a filter — preserves the coverage-gap framing.

COSMoS coverage aggregations (`DSS_Count`, `BC_Count`, `DS_Codes`, `BC_IDs`)
are computed over scope-domain DSSs only, not graph-wide.

In [4]:
# Filter SDTM_Test_Identity to TESTCDs that participate in any immunogenicity-scope domain.
def in_scope(domains_str):
    if not domains_str:
        return False
    doms = {d.strip() for d in domains_str.split(';')}
    return bool(doms & set(SCOPE_DOMAINS))

ref_in_scope = ref_tests[ref_tests['SDTM_Domains'].apply(in_scope)].copy()
print(f"Immunogenicity-scoped TESTCDs: {len(ref_in_scope):,}")

# In_Scope_Domains: filter SDTM_Domains down to only the scope members for each row.
def filter_to_scope(domains_str):
    doms = sorted({d.strip() for d in domains_str.split(';') if d.strip() in SCOPE_DOMAINS})
    return '; '.join(doms)

ref_in_scope['In_Scope_Domains'] = ref_in_scope['SDTM_Domains'].apply(filter_to_scope)

# Per-domain breakdown
print('Per-domain TESTCD count:')
for d in SCOPE_DOMAINS:
    n = ref_in_scope['SDTM_Domains'].apply(lambda s: d in [x.strip() for x in s.split(';')]).sum()
    print(f'  {d}: {n:>5,}')

Immunogenicity-scoped TESTCDs: 100
Per-domain TESTCD count:
  IS:   100


In [5]:
# COSMoS coverage from DSS_View Measurement_Specs, filtered to scope domains.
# This computes per-TESTCD aggregations over scope DSSs only — graph-wide aggregations
# from view_ti would over-count when a TESTCD also pins to non-scope domains.
view_ms_scope = view_ms[view_ms['domain'].isin(SCOPE_DOMAINS)].copy()
print(f"Immunogenicity-scope DSS rows in DSS_View: {len(view_ms_scope):,}")

def _join_unique(s):
    return '; '.join(sorted({v for v in s if v}))

# Keyed on TESTCD + the test's NCIt code: TESTCD alone is a mnemonic, reused across
# domains for unrelated tests (MCH in LB vs MB, MV in MK vs RE).
cosmos_cov = view_ms_scope.groupby(['TESTCD_value', 'TESTCD_ncit'], sort=True).agg(
    DSS_Count=('ds_id', 'nunique'),
    BC_Count=('bc_id', 'nunique'),
    DS_Codes=('ds_id', _join_unique),
    BC_IDs=('bc_id', _join_unique),
).reset_index().rename(columns={'TESTCD_value': 'TESTCD', 'TESTCD_ncit': 'NCIt_Code'})

cosmos_cov['DSS_Count'] = cosmos_cov['DSS_Count'].astype(str)
cosmos_cov['BC_Count'] = cosmos_cov['BC_Count'].astype(str)
print(f"COSMoS-pinned TESTCDs in scope: {len(cosmos_cov):,}")

# Conflict flags from DSS_View Test_Identity, keyed on TESTCD + NCIt code as cosmos_cov:
# a flag on a graph TESTCD says nothing about a same-mnemonic test with another NCIt code.
flags = view_ti[['TESTCD', 'NCIt_Code', 'NCIt_Code_Conflict', 'NCIt_Reference_Disagree']].copy()

Immunogenicity-scope DSS rows in DSS_View: 326
COSMoS-pinned TESTCDs in scope: 7


In [6]:
ti = ref_in_scope.copy()

# Has_DSS: derived membership in cosmos_cov (i.e. COSMoS pins this TESTCD in any scope domain).
_cov_keys = set(zip(cosmos_cov['TESTCD'], cosmos_cov['NCIt_Code']))
ti['Has_DSS'] = [(t, n) in _cov_keys for t, n in zip(ti['TESTCD'], ti['NCIt_Code'])]
ti['Has_DSS'] = ti['Has_DSS'].map({True: 'Yes', False: 'No'})

# Join COSMoS coverage. Default counts to '0' for non-pinned TESTCDs.
ti = ti.merge(cosmos_cov, on=['TESTCD', 'NCIt_Code'], how='left').fillna('')
ti.loc[ti['DSS_Count'] == '', 'DSS_Count'] = '0'
ti.loc[ti['BC_Count'] == '', 'BC_Count'] = '0'

# Join NCIt conflict flags. Empty for non-pinned TESTCDs (only pinned TESTCDs can disagree).
ti = ti.merge(flags, on=['TESTCD', 'NCIt_Code'], how='left').fillna('')
# 'Not evaluated' where the flag was not computed (see the ReadMe for the two causes).
for _c in ['NCIt_Code_Conflict', 'NCIt_Reference_Disagree']:
    ti.loc[ti[_c] == '', _c] = 'Not evaluated'

# Final column order — keys, reference identity (green), COSMoS coverage (yellow), flags (grey).
TI_COLS = [
    'TESTCD', 'NCIt_Code', 'In_Scope_Domains', 'SDTM_Domains',
    'TEST', 'NCIt_Preferred_Term', 'NCIt_Synonyms', 'NCIt_Definition',
    'UMLS_CUI', 'NCIm_CUI',
    'Has_DSS', 'DSS_Count', 'BC_Count', 'DS_Codes', 'BC_IDs',
    'NCIt_Code_Conflict', 'NCIt_Reference_Disagree',
]

missing = [c for c in TI_COLS if c not in ti.columns]
if missing:
    raise RuntimeError(f'Missing expected Test_Identity columns: {missing}')

ti_final = ti[TI_COLS].copy()
print(f"Test_Identity: {len(ti_final):,} rows x {len(ti_final.columns)} cols")
print(f"  With COSMoS coverage (Has_DSS=Yes): {(ti_final['Has_DSS'] == 'Yes').sum():,}")
print(f"  Coverage gap   (Has_DSS=No):        {(ti_final['Has_DSS'] == 'No').sum():,}")
print(f"  NCIt_Code_Conflict=Yes:             {(ti_final['NCIt_Code_Conflict'] == 'Yes').sum()}")
print(f"  NCIt_Reference_Disagree=Yes:        {(ti_final['NCIt_Reference_Disagree'] == 'Yes').sum()}")

Test_Identity: 100 rows x 17 cols
  With COSMoS coverage (Has_DSS=Yes): 7
  Coverage gap   (Has_DSS=No):        93
  NCIt_Code_Conflict=Yes:             0
  NCIt_Reference_Disagree=Yes:        0


## 4. Build Measurement_Specs

Filter `DSS_View.Measurement_Specs` to immunogenicity-scope domains. Add:

- `Observation_Class` join from `SDTM_Domain_Metadata`.
- `LOINC_DSS` (= `LOINC_value`, DSS-grain pin — primary).
- `LOINC_BC` (coalesce of `Coding_http://loinc.org/` and `Coding_https://loinc.org`
  — BC-grain, with the source-data URI inconsistency flattened).

Validation checks (TESTCD_NCIt ↔ BC NCIt consistency, Quantitative-without-units,
etc.) belong in a separate validation step, not in this consumer projection.
See `cosmos-bc-dss/notebooks/COSMoS_BC_DSS_Validate.ipynb` for the legacy QC pattern.

Rename `TESTCD_value` → `TESTCD`, `TEST_value` → `TEST`, `TESTCD_ncit` → `TESTCD_NCIt`
for join symmetry with the `Test_Identity` sheet. Other slot columns keep their
`<remainder>_value/_ncit/_codelist/_value_list/_value_list_ncit` 5-tuple naming
(value_list and pin are mutually exclusive in COSMoS source — see DSS_View ReadMe).

In [7]:
ms = view_ms[view_ms['domain'].isin(SCOPE_DOMAINS)].copy()
print(f"Measurement_Specs rows in scope: {len(ms):,}")
print(f"  Domains present:           {sorted(ms['domain'].unique())}")
print(f"  Domains with no DSSs yet:  {sorted(set(SCOPE_DOMAINS) - set(ms['domain'].unique()))}")

Measurement_Specs rows in scope: 326
  Domains present:           ['IS']
  Domains with no DSSs yet:  []


In [8]:
# Observation_Class join.
ms = ms.merge(
    domain_meta[['Domain', 'Observation_Class']].rename(columns={'Domain': 'domain'}),
    on='domain', how='left',
  ).fillna('')

# LOINC_BC: coalesce the two source URI variants. The graph projects the URI
# inconsistency in the BC Coding list verbatim (see cosmos-graph/docs/COSMoS_Graph.md).
LOINC_BC_SOURCES = ['Coding_http://loinc.org/', 'Coding_https://loinc.org']
def _coalesce_loinc(row):
    parts = set()
    for col in LOINC_BC_SOURCES:
        v = row.get(col, '')
        if v:
            for p in v.split(';'):
                p = p.strip()
                if p:
                    parts.add(p)
    return '; '.join(sorted(parts))

ms['LOINC_BC'] = ms.apply(_coalesce_loinc, axis=1)
print(f'LOINC_BC populated rows: {(ms["LOINC_BC"] != "").sum()}')

LOINC_BC populated rows: 0


In [9]:
# Allowed_Units: conditional expansion of ORRESU_codelist when the codelist is narrow
# enough that per-row expansion is signal, not noise. Kept for shape parity with the
# other Findings consumers; the ReadMe states how many rows it fires on.
# Kept for shape parity with Measurement_Findings, where VSRESU (~29 terms) does fire.
codelist_size = ct_terms.groupby('codelist_submission_value')['term_submission_value'].nunique()
narrow_codelists = set(codelist_size[codelist_size <= ALLOWED_UNITS_THRESHOLD].index)
print(f'Codelists with <= {ALLOWED_UNITS_THRESHOLD} terms: {len(narrow_codelists):,}')

terms_by_codelist = {}
for cl, group in ct_terms.groupby('codelist_submission_value'):
    if cl in narrow_codelists:
        terms = sorted({t for t in group['term_submission_value'] if t})
        terms_by_codelist[cl] = '; '.join(terms)

ms['Allowed_Units'] = ms['ORRESU_codelist'].map(terms_by_codelist).fillna('')
expanded = (ms['Allowed_Units'] != '').sum()
print(f'Rows with Allowed_Units expansion: {expanded:,}')
if expanded:
    fired_codelists = sorted({cl for cl in ms['ORRESU_codelist'] if cl in narrow_codelists})
    print(f'  Fired codelists: {fired_codelists}')

Codelists with <= 50 terms: 250
Rows with Allowed_Units expansion: 0


In [10]:
# Rename DSS_View native columns to consumer link-key naming.
# Local exception to the leverage-DSS_View-naming rule, for join symmetry with Test_Identity.
ms = ms.rename(columns={
    'TESTCD_value': 'TESTCD',
    'TEST_value':   'TEST',
    'TESTCD_ncit':  'TESTCD_NCIt',
    'LOINC_value':  'LOINC_DSS',
})

In [11]:
# SLOT_SCHEMA declares which slot remainders this consumer emits, classified by policy:
#   core     — column always emitted, even when no row fires (empty column).
#              Reserved for first-order structural slots (decomposition axes).
#   optional — column emitted only when at least one row fires.
#              Includes qualifier slots (units, location, fasting status,
#              result category, evaluator, test detail), whose presence is
#              a consequence of other columns (e.g. ORRESU/STRESU presence
#              is conditional on result_scales including Quantitative).
#   exclude  — slot handled elsewhere (link-key, LOINC).
# A "slot" covers both pinned variables (assigned_term_value/_ncit) and
# value_list-restricted variables. Pin and value_list are mutually exclusive
# in COSMoS source — see DSS_View ReadMe. Drift detection: slots firing in
# source but not declared get a WARN at runtime.
# Binding agent is core: in IS it is the axis the DSSs of one BC differ by (Consumer_Exclusions IS-1),
# and the IG reads TEST + BDAGNT as the analyte (SDTMIG v3.4 ISBDAGNT CDISC Notes).
SLOT_SCHEMA = {
    'core':     ['BDAGNT', 'SPEC', 'METHOD'],
    'optional': ['TSTDTL', 'TSTOPO', 'CAT', 'ORRES', 'ORRESU', 'STRESC', 'STRESN', 'STRESU',
                 'LOC', 'EVAL', 'FAST', 'RESCAT'],
    'exclude':  ['TESTCD', 'TEST', 'LOINC'],
}

SLOT_SUFFIXES = ('_value', '_ncit', '_codelist', '_value_list', '_value_list_ncit')


def fires(remainder):
    for suffix in SLOT_SUFFIXES:
        col = f'{remainder}{suffix}'
        if col in ms.columns and (ms[col] != '').any():
            return True
    return False


declared = set(SLOT_SCHEMA['core']) | set(SLOT_SCHEMA['optional']) | set(SLOT_SCHEMA['exclude'])

# Candidate remainders: any slot with a _value or _value_list column in source
all_candidate_remainders = sorted(
    {c.rsplit('_value_list_ncit', 1)[0] for c in ms.columns if c.endswith('_value_list_ncit')} |
    {c.rsplit('_value_list', 1)[0] for c in ms.columns if c.endswith('_value_list')} |
    {c.rsplit('_value', 1)[0] for c in ms.columns if c.endswith('_value') and not c.endswith('_value_list')}
)

slots_to_emit = []
notes = []

# Core: always emit, in declared order. Warn if no columns present in source.
for r in SLOT_SCHEMA['core']:
    if f'{r}_value' not in ms.columns and f'{r}_value_list' not in ms.columns:
        notes.append(f"WARN: core slot '{r}' has no columns in DSS_View — schema likely needs updating.")
        continue
    slots_to_emit.append(r)
    if not fires(r):
        notes.append(f"INFO: core slot '{r}' not firing on this scope — schema preserved, column will be empty.")

# Optional: emit only when firing, in declared order.
for r in SLOT_SCHEMA['optional']:
    if fires(r):
        slots_to_emit.append(r)

# Undeclared firing slots: WARN and emit at end (drift detection).
for r in all_candidate_remainders:
    if r in declared:
        continue
    if fires(r):
        col = f'{r}_value' if f'{r}_value' in ms.columns else f'{r}_value_list'
        n_rows = (ms[col] != '').sum()
        notes.append(f"WARN: undeclared slot '{r}' firing on {n_rows} rows — add to SLOT_SCHEMA core/optional/exclude.")
        slots_to_emit.append(r)

for note in notes:
    print(note)
if notes:
    print()

print(f"Slots to emit ({len(slots_to_emit)}, in order): {slots_to_emit}")
print(f"  Core (always present):  {[r for r in slots_to_emit if r in SLOT_SCHEMA['core']]}")
print(f"  Optional (firing-only): {[r for r in slots_to_emit if r in SLOT_SCHEMA['optional']]}")
print(f"  Undeclared (firing):    {[r for r in slots_to_emit if r not in declared]}")


Slots to emit (10, in order): ['BDAGNT', 'SPEC', 'METHOD', 'TSTDTL', 'TSTOPO', 'ORRES', 'ORRESU', 'STRESC', 'STRESN', 'STRESU']
  Core (always present):  ['BDAGNT', 'SPEC', 'METHOD']
  Optional (firing-only): ['TSTDTL', 'TSTOPO', 'ORRES', 'ORRESU', 'STRESC', 'STRESN', 'STRESU']
  Undeclared (firing):    []


In [12]:
# Assemble final column order — keys, test identity, BC identity, DSS identity,
# firing-slot blocks, LOINC pair, Allowed_Units, conflict flag.
identity_cols = [
    'domain', 'Observation_Class', 'ds_id', 'bc_id', 'TESTCD',
    'TEST', 'TESTCD_NCIt',
    'bc_short_name', 'bc_definition', 'bc_categories', 'bc_parent_label',
    'bc_hierarchy_path', 'bc_type', 'result_scales', 'bc_ncit_code',
    'ds_short_name', 'sdtmig_start_version', 'sdtmig_end_version',
    'package_date', 'source',
]

slot_cols = []
for r in slots_to_emit:
    for suffix in SLOT_SUFFIXES:
        col = f'{r}{suffix}'
        if col in ms.columns:
            slot_cols.append(col)

# A _codelist column empty on every row of this file says nothing: leave it out.
# Core slots keep their full column set.
slot_cols = [c for c in slot_cols
             if not (c.endswith('_codelist') and (ms[c] == '').all()
                     and c[:-len('_codelist')] not in SLOT_SCHEMA['core'])]

loinc_cols = ['LOINC_DSS', 'LOINC_BC']
tail_cols = ['Allowed_Units']

MS_COLS = identity_cols + slot_cols + loinc_cols + tail_cols

missing = [c for c in MS_COLS if c not in ms.columns]
if missing:
    raise RuntimeError(f'Missing expected Measurement_Specs columns: {missing}')

ms_final = ms[MS_COLS].copy()
print(f"Measurement_Specs: {len(ms_final):,} rows x {len(ms_final.columns)} cols")
print(f"  Identity: {len(identity_cols)}  Slots: {len(slot_cols)}  LOINC: {len(loinc_cols)}  Tail: {len(tail_cols)}")

Measurement_Specs: 326 rows x 62 cols
  Identity: 20  Slots: 39  LOINC: 2  Tail: 1


## 4b. Axis_Meaning

Copies the rows of `Variable_Meaning` (`SDTM_Domain_Metadata.xlsx`) that apply to this file:
rows whose `Domain` has DSSs in `Measurement_Specs`, and the axis rows (`Domain` empty; today
only scale). These rows are our position (Layer B), not a published fact. `Position`, `Basis` and
`Status` are copied as written, and `Basis_Note` with them, so the file states its own basis.
`Columns` names the `Measurement_Specs` columns the row refers to. A variable without a row: no
position taken.

No observable key is built from these rows in this file (Phase 2 key: LB, MB, MI only).

The full check of `Variable_Meaning` is in `domain-behaviour` (section 4f), which runs later.
This cell stops if the sheet's columns are not the expected ones, if an axis row has no column
block here, or if a row's columns are not in `Measurement_Specs`.


In [13]:
MEANING_COLUMNS = ['Key_Source', 'Domain', 'Key', 'Position', 'Basis', 'Basis_Note', 'Status']
# Measurement_Specs column blocks for the axis rows (Domain empty in Variable_Meaning).
AXIS_BLOCKS = {'scale': ['ORRES', 'ORRESU']}

meaning = pd.read_excel(DOMAIN_META_FILE, sheet_name='Variable_Meaning', dtype=str).fillna('')
if list(meaning.columns) != MEANING_COLUMNS:
    raise ValueError(f'Variable_Meaning columns {list(meaning.columns)}, expected {MEANING_COLUMNS}')

axis_meaning = meaning[meaning['Domain'].isin(set(ms_final['domain'])) | (meaning['Domain'] == '')].copy()

unknown = sorted(set(axis_meaning.loc[axis_meaning['Domain'] == '', 'Key']) - set(AXIS_BLOCKS))
if unknown:
    raise ValueError(f'Variable_Meaning axis rows without a column block in this notebook: {unknown}')


def columns_for(r):
    blocks = AXIS_BLOCKS[r['Key']] if r['Domain'] == '' else [r['Key'][2:]]
    return '; '.join(c for c in ms_final.columns if any(c == b or c.startswith(b + '_') for b in blocks))


axis_meaning['Columns'] = axis_meaning.apply(columns_for, axis=1)
bad = axis_meaning[axis_meaning['Columns'] == '']
if len(bad):
    raise ValueError(f'Variable_Meaning rows whose columns are not in Measurement_Specs:\n{bad[["Domain", "Key"]].to_string(index=False)}')

axis_meaning = axis_meaning[['Domain', 'Key', 'Columns', 'Position', 'Basis', 'Basis_Note', 'Status']]
print(f'Axis_Meaning: {len(axis_meaning)} rows')
print(axis_meaning[['Domain', 'Key', 'Position', 'Status', 'Columns']].to_string(index=False))


Axis_Meaning: 6 rows
Domain      Key  Position   Status                                                                                                                                                 Columns
    IS   ISSPEC    system accepted                                                                             SPEC_value; SPEC_ncit; SPEC_codelist; SPEC_value_list; SPEC_value_list_ncit
    IS ISMETHOD    method accepted                                                                   METHOD_value; METHOD_ncit; METHOD_codelist; METHOD_value_list; METHOD_value_list_ncit
    IS ISTSTDTL      open proposed                                                                                                              TSTDTL_value; TSTDTL_ncit; TSTDTL_codelist
    IS ISBDAGNT component accepted                                                                                                              BDAGNT_value; BDAGNT_ncit; BDAGNT_codelist
    IS ISTSTOPO      open proposed          

## Codelists and shared strings

Built here, before the workbook is written, so the ReadMe can state the counts.

`Codelists`: every codelist named in a `*_codelist` column of `Measurement_Specs`, with all its
terms, from COSMoS_Graph_CT (SDTM CT as resolved for the graph). The set is derived from the
file's own columns, not chosen by hand. `term_synonyms` = CDISC synonyms (SDTM CT, part of the
standard). `term_ncit_synonyms` = the NCIt synonyms of the term's NCIt code that are not already
its submission value, NCI preferred term or a CDISC synonym, compared lower-cased, letters and
digits only (`sdtm-test-codes/interim/SDTM_CT_Term_NCIt_Synonyms.csv`, NCI EVS FLAT file).
Separated by `|`, as in the NCIt FLAT file: some NCIt synonyms contain `;`.

`Shared_Strings`: one row per string and term, for every string, in that comparison form, that names
more than one term within one codelist. `Shared_Via` = *CT fields* when two or more terms carry it in their SDTM CT fields
(submission value, NCI preferred term, CDISC synonym); *NCIt synonyms* when the sharing comes only
with `term_ncit_synonyms`. A string that names more than one term identifies none of them.

The cell stops if the synonym file is missing, if its FLAT date is not the `NCIt` pin, or if a
term of the sheet has no row in it.

In [14]:
import re

TERM_NCIT_FILE = REPO_ROOT / 'sdtm-test-codes' / 'interim' / 'SDTM_CT_Term_NCIt_Synonyms.csv'
if not TERM_NCIT_FILE.exists():
    raise FileNotFoundError(f'SDTM_CT_Term_NCIt_Synonyms not found: {TERM_NCIT_FILE}')
term_ncit = pd.read_csv(TERM_NCIT_FILE, dtype=str).fillna('')
if set(term_ncit['NCIt_FLAT_Date']) != {VERSIONS['NCIt']}:
    raise RuntimeError(f"NCIt pin {VERSIONS['NCIt']} != FLAT date {sorted(set(term_ncit['NCIt_FLAT_Date']))} "
                       f"in {TERM_NCIT_FILE.name}: re-run SDTM_CT_Term_NCIt_Synonyms")
ncit_syn_of = dict(zip(term_ncit['NCIt_Code'], term_ncit['NCIt_Synonyms']))

# Every codelist named in a *_codelist column of Measurement_Specs, with all its terms,
# from COSMoS_Graph_CT. A reader resolves any codelist reference in this file without
# opening another file. The set is derived from the file's own columns, not chosen by hand.
ct_codelists = pd.read_excel(GRAPH_CT_FILE, sheet_name='Codelists', dtype=str).fillna('')

CL_REF_COLS = [c for c in ms_final.columns if c.endswith('_codelist')]
cl_refs = {}
for c in CL_REF_COLS:
    for v in ms_final[c].fillna(''):
        if v:
            cl_refs.setdefault(v, set()).add(c)

missing_cl = sorted(set(cl_refs) - set(ct_terms['codelist_submission_value']))
if missing_cl:
    raise RuntimeError(f'Codelists referenced in Measurement_Specs but not in COSMoS_Graph_CT CodelistTerms: {missing_cl}')

codelists_sheet = ct_terms[ct_terms['codelist_submission_value'].isin(cl_refs)].merge(
    ct_codelists[['codelist_concept_id', 'codelist_name', 'codelist_extensible']],
    on='codelist_concept_id', how='left', validate='many_to_one',
)
codelists_sheet['referenced_by'] = codelists_sheet['codelist_submission_value'].map(
    lambda v: '; '.join(sorted(cl_refs[v])))

_no_ncit = sorted(set(codelists_sheet['term_concept_id']) - set(ncit_syn_of))
if _no_ncit:
    raise RuntimeError(f'Codelist terms without a row in {TERM_NCIT_FILE.name}: {_no_ncit[:10]}')

_TOKEN = re.compile(r'[A-Za-z0-9]+')


def norm(text):
    """Comparison form: lower case, letters and digits only."""
    return ' '.join(_TOKEN.findall(text.lower()))


def split_syn(text, sep=';'):
    return [s.strip() for s in text.split(sep) if s.strip()]


def ct_strings(r):
    return [r['term_submission_value'], r['term_nci_preferred_term'], *split_syn(r['term_synonyms'])]


def new_ncit_synonyms(r):
    have = {norm(s) for s in ct_strings(r)}
    kept = []
    for s in split_syn(ncit_syn_of[r['term_concept_id']], '|'):
        k = norm(s)
        if k and k not in have:
            have.add(k)
            kept.append(s)
    return '|'.join(kept)


codelists_sheet['term_ncit_synonyms'] = codelists_sheet.apply(new_ncit_synonyms, axis=1)
CL_COLS = [
    'codelist_submission_value', 'codelist_concept_id', 'codelist_name', 'codelist_extensible',
    'term_submission_value', 'term_concept_id', 'term_nci_preferred_term',
    'term_synonyms', 'term_ncit_synonyms', 'term_definition', 'referenced_by',
]
codelists_sheet = codelists_sheet[CL_COLS].sort_values(
    ['codelist_submission_value', 'term_submission_value']).reset_index(drop=True)

# Shared strings: one occurrence per (codelist, string, term); CT fields win over NCIt synonyms.
_occ = []
for _, r in codelists_sheet.iterrows():
    for source, values in (('CT', ct_strings(r)), ('NCIt', split_syn(r['term_ncit_synonyms'], '|'))):
        for v in values:
            if norm(v):
                _occ.append((r['codelist_submission_value'], norm(v), r['term_submission_value'],
                             r['term_concept_id'], source))
_occ = (pd.DataFrame(_occ, columns=['codelist_submission_value', 'String', 'term', 'concept', 'source'])
        .sort_values('source').drop_duplicates(['codelist_submission_value', 'String', 'concept']))
_n_terms = _occ.groupby(['codelist_submission_value', 'String'])['concept'].transform('nunique')
_occ = _occ[_n_terms > 1].copy()
# One row per string and term: term names can contain commas ("LIMB, UPPER"), so they are not packed.
_by_string = _occ.groupby(['codelist_submission_value', 'String'])
_occ['Term_Count'] = _by_string['concept'].transform('nunique').astype(str)
_occ['Shared_Via'] = _by_string['source'].transform(lambda s: 'CT fields' if (s == 'CT').sum() > 1 else 'NCIt synonyms')
shared_strings = (_occ.rename(columns={'term': 'term_submission_value', 'concept': 'term_concept_id', 'source': 'Source'})
                  [['codelist_submission_value', 'String', 'Term_Count', 'Shared_Via',
                    'term_submission_value', 'term_concept_id', 'Source']]
                  .sort_values(['codelist_submission_value', 'String', 'term_submission_value', 'term_concept_id'])
                  .reset_index(drop=True))
_strings = shared_strings.drop_duplicates(['codelist_submission_value', 'String'])
n_shared_strings = len(_strings)
shared_by_cl = (_strings.groupby(['codelist_submission_value', 'Shared_Via']).size()
                .unstack(fill_value=0)
                .reindex(index=sorted(codelists_sheet['codelist_submission_value'].unique()),
                         columns=['CT fields', 'NCIt synonyms'], fill_value=0))
print(f"Codelists: {codelists_sheet['codelist_submission_value'].nunique()} codelists, {len(codelists_sheet):,} terms; "
      f"{(codelists_sheet['term_ncit_synonyms'] != '').sum():,} with term_ncit_synonyms")
print(f'Shared_Strings: {n_shared_strings:,} strings, {len(shared_strings):,} rows (string x term)')
print(shared_by_cl.to_string())

Codelists: 7 codelists, 3,679 terms; 1,149 with term_ncit_synonyms
Shared_Strings: 109 strings, 244 rows (string x term)
Shared_Via                 CT fields  NCIt synonyms
codelist_submission_value                          
ISBDAGT                            2              4
ISFTSDTL                           0              0
METHOD                             1              5
MICROORG                           1             11
SPECTYPE                           0              0
TSTOPOBJ                           0              0
UNIT                              40             45


## 5. Write workbook

Five sheets: `ReadMe`, `Test_Identity`, `Measurement_Specs`, `Axis_Meaning`, `Codelists`.

Header colour convention (per repo standard):

- **Green** — TESTCD / SDTM-CT-side columns (NCI EVS reference identity), and `Codelists`.
- **Yellow** — COSMoS-side columns (BC, DSS, slot pivot — pin and value_list).
- **Grey** — keys, conflict flags, aggregation columns.
- **Blue** — `Axis_Meaning`: our position, copied from `Variable_Meaning` (blue there too).


In [15]:
HEADER_FONT = Font(name='Arial', bold=True, size=10, color='FFFFFF')
DATA_FONT = Font(name='Arial', size=10)
WRAP = Alignment(wrap_text=True, vertical='top')

GREEN_HEADER  = PatternFill('solid', fgColor='548235')
YELLOW_HEADER = PatternFill('solid', fgColor='FFD700')
GREY_HEADER   = PatternFill('solid', fgColor='808080')


def write_sheet(ws, df, header_fills, col_widths):
    """Write a DataFrame; header_fills maps col_name -> PatternFill;
    col_widths maps col_name -> width (default 18 if missing)."""
    cols = list(df.columns)
    for ci, name in enumerate(cols, 1):
        cell = ws.cell(row=1, column=ci, value=name)
        cell.font = HEADER_FONT
        cell.fill = header_fills.get(name, GREY_HEADER)
        cell.alignment = WRAP
    for ri, (_, row) in enumerate(df.iterrows(), 2):
        for ci, name in enumerate(cols, 1):
            val = row[name]
            cell = ws.cell(row=ri, column=ci, value=val if val != '' else None)
            cell.font = DATA_FONT
            cell.alignment = WRAP
    for ci, name in enumerate(cols, 1):
        ws.column_dimensions[get_column_letter(ci)].width = col_widths.get(name, 18)
    ws.freeze_panes = 'A2'
    ws.auto_filter.ref = f'A1:{get_column_letter(len(cols))}1'

print('Style helpers ready.')

Style helpers ready.


In [16]:
wb = Workbook()

# ── ReadMe sheet ──
ws_rm = wb.active
ws_rm.title = 'ReadMe'

readme_font = Font(name='Arial', size=10)
title_font = Font(name='Arial', size=12, bold=True)
section_font = Font(name='Arial', size=10, bold=True)

# Values for the ReadMe, computed so the text follows the content.
_ti_domains = {d for v in ti_final['In_Scope_Domains'] for d in v.split('; ') if d}
_no_testcd = sorted(set(SCOPE_DOMAINS) - _ti_domains)
if _no_testcd:
    raise RuntimeError(f'In-scope domains without SDTM_Test_Identity rows: {_no_testcd}: add a KNOWN GAPS line')
_no_dss = sorted((set(SCOPE_DOMAINS) & _ti_domains) - set(ms_final['domain']))
# BCs with DSSs, and the source codelist of each assigned binding agent.
_bcs = ms_final.groupby(['bc_id', 'bc_short_name']).size().reset_index(name='n').sort_values('bc_short_name')
_bdagnt_cl = ms_final['BDAGNT_codelist'].value_counts()
_bdagnt_none = int((ms_final['BDAGNT_value'] == '').sum())
_loinc_rows = int(((ms_final['LOINC_DSS'] != '') | (ms_final['LOINC_BC'] != '')).sum())
# The tier variable (--TSTOPO): value list only; its bound codelist comes from DSS_View TSTOPO_codelist.
_tstopo_cl = sorted(set(ms_final.loc[ms_final['TSTOPO_value_list'] != '', 'TSTOPO_codelist']))
if _tstopo_cl != ['TSTOPOBJ']:
    raise RuntimeError(f'TSTOPO value lists bound to {_tstopo_cl}, expected [TSTOPOBJ]: revise the TSTOPO ReadMe lines')
_tstopo_terms = sorted(codelists_sheet.loc[codelists_sheet['codelist_submission_value'] == 'TSTOPOBJ', 'term_submission_value'])
_tstopo_dss = int((ms_final['TSTOPO_value_list'] != '').sum()) if 'TSTOPO_value_list' in ms_final.columns else 0
_ada = ti_final[ti_final['TEST'].str.contains('Antidrug', case=False)]

readme_lines = [
    ('Immunogenicity_Findings — graph-fed consumer', title_font),
    ('', None),
    ('PURPOSE', section_font),
    ('Immunogenicity Findings for study design, SoA-to-CDISC mapping and USDM: SDTM CT test identity', readme_font),
    ('  (Test_Identity) with the COSMoS Dataset Specializations that specify it (Measurement_Specs),', readme_font),
    ('  and the binding agents they name (Codelists: ISBDAGT, MICROORG).', readme_font),
    ('', None),
    ('SCOPE', section_font),
    ('Immunogenicity Specimen Assessments (SDTMIG v3.4 section 6.3.5.5): antigen-induced humoral or', readme_font),
    ('  cell-mediated immune response in the subject.', readme_font),
    (f'In scope: {", ".join(SCOPE_DOMAINS)} (SDTM_Domain_Metadata Immunogenicity = Yes, minus exclusions).', readme_font),
    (f'Excluded: {", ".join(EXCLUDED_DOMAINS) or "none"} (SDTM_Domain_Metadata Consumer_Exclusions).', readme_font),
    *[(f'Not in Specimen_Findings ({claims}): {why}', readme_font) for d, (claims, why) in SPECIMEN_EXCLUSION.items()],
    ('', None),
    ('SOURCE VERSIONS', section_font),
    (f'SDTM CT {VERSIONS["SDTM_CT"]} · COSMoS {VERSIONS["COSMoS"]} · SDTMIG v{VERSIONS["SDTMIG"]} / SDTM v{VERSIONS["SDTM"]} · NCIt FLAT {VERSIONS["NCIt"]}', readme_font),
    ('  Pinned in Reference_Versions.json (repo root). COSMoS is checked: it must equal the newest', readme_font),
    ('  package_date in DSS_View. The SDTM CT file carries no version, so that pin is stated, not checked.', readme_font),
    ('  NCIt: the build date of the NCI EVS FLAT file (it carries no release number); checked against', readme_font),
    ('  SDTM_CT_Term_NCIt_Synonyms.csv.', readme_font),
    (f'Generated: {datetime.now():%Y-%m-%d %H:%M}', readme_font),
    ('Notebook: sdtm-findings-graph/notebooks/Immunogenicity_Findings.ipynb', readme_font),
    ('Track:    sdtm-findings-graph (graph-fed Findings consumer)', readme_font),
    ('Inputs:', readme_font),
    ('  consumer-bases/interim/DSS_View.xlsx (Test_Identity, Measurement_Specs)', readme_font),
    ('  sdtm-test-codes/machine_actionable/SDTM_Test_Identity.xlsx (Test Codes)', readme_font),
    ('  sdtm-domain-reference/machine_actionable/SDTM_Domain_Metadata.xlsx (Domains, Consumer_Exclusions, Variable_Meaning)', readme_font),
    ('  cosmos-graph/interim/COSMoS_Graph_CT.xlsx (Codelists, CodelistTerms — for Allowed_Units and the Codelists sheet)', readme_font),
    ('    An interim pipeline file: rebuilt at every SDTM CT and COSMoS release, its layout may change.', readme_font),
    ('    Use the Codelists sheet here, not that file.', readme_font),
    ('  sdtm-test-codes/interim/SDTM_CT_Term_NCIt_Synonyms.csv (NCIt synonyms — for term_ncit_synonyms)', readme_font),
    ('', None),
    ('KNOWN GAPS', section_font),
    (f'{(ti_final["Has_DSS"] == "No").sum():,} of {len(ti_final):,} Test_Identity rows are concept identity only (Has_DSS = No).', readme_font),
    (f'COSMoS DSSs exist for {len(_bcs)} BCs only:', readme_font),
    *[(f'  {r.bc_short_name} ({r.bc_id}): {r.n} DSSs', readme_font) for r in _bcs.itertuples()],
    (f'Antidrug antibody tests: {len(_ada)} Test_Identity rows, {(_ada["Has_DSS"] == "Yes").sum()} with a DSS. For these tests the binding', readme_font),
    ('  agent is the study therapy, which CDISC does not control: sponsors extend ISBDAGT in Define-XML', readme_font),
    ('  (SDTMIG v3.4 IS Assumption 7a). The Codelists sheet has no term for it by design.', readme_font),
    ('Binding agent: SDTMIG v3.4 binds --BDAGNT to two codelists, MICROORG and ISBDAGT (IS Assumption 7);', readme_font),
    ('  the CDISC "Controlled Terminology Rules for Immunogenicity Tests" say which to use. Not read for this file.', readme_font),
    (f'  BDAGNT_codelist names the codelist of each assigned term: {", ".join(f"{k} {v}" for k, v in _bdagnt_cl.items())} DSSs;', readme_font),
    (f'  {_bdagnt_none} without a binding agent.', readme_font),
    ('Neighbouring domains (SDTMIG v3.4 IS Assumptions 4–6): flow cytometry data go to CP; cytokines,', readme_font),
    ('  chemokines and complement proteins to LB; microbial antigen/antibody combination tests to MB.', readme_font),
    ('  A binding agent or microorganism named in an activity does not by itself place it in IS.', readme_font),
    (f'Test Operational Objective (--TSTOPO; SCREEN, CONFIRM, QUANTIFY): a value list on {_tstopo_dss} DSSs, never assigned.', readme_font),
    ('  Whether it is part of the observable is open (Axis_Meaning). Its codelist TSTOPOBJ is in the Codelists sheet', readme_font),
    (f'  ({len(_tstopo_terms)} terms in SDTM CT: {", ".join(_tstopo_terms)}); SDTMIG v3.4 IS Assumption 8 lists SCREEN,', readme_font),
    ('  CONFIRM and QUANTIFY and calls the codelist nonextensible.', readme_font),
    (f'LOINC: {_loinc_rows} of {len(ms_final):,} Measurement_Specs rows carry a LOINC code; the columns are kept for shape parity.', readme_font),
    (f'Conflict flags evaluated: NCIt_Code_Conflict on {(ti_final["NCIt_Code_Conflict"] != "Not evaluated").sum():,}, NCIt_Reference_Disagree on', readme_font),
    (f'  {(ti_final["NCIt_Reference_Disagree"] != "Not evaluated").sum():,} of {len(ti_final):,} rows; rows flagged Yes in either: {int((ti_final[["NCIt_Code_Conflict", "NCIt_Reference_Disagree"]] == "Yes").any(axis=1).sum()):,}.', readme_font),
    *([(f'In scope without DSSs (Test_Identity only): {", ".join(_no_dss)}.', readme_font)] if _no_dss else []),
    (f'Allowed_Units is empty on {(ms_final["Allowed_Units"].fillna("") == "").sum():,} of {len(ms_final):,} Measurement_Specs rows: the units codelist has', readme_font),
    (f'  more than {ALLOWED_UNITS_THRESHOLD} terms, or none is bound. The terms are in sheet Codelists.', readme_font),
    (f'Shared strings (sheet Shared_Strings): {n_shared_strings:,} strings name more than one term within one codelist', readme_font),
    (f'  of the Codelists sheet: {int(shared_by_cl["CT fields"].sum()):,} already in the SDTM CT fields, {int(shared_by_cl["NCIt synonyms"].sum()):,} only with', readme_font),
    ('  term_ncit_synonyms. Per codelist (CT fields / NCIt synonyms):', readme_font),
    *[(f'    {cl}: {int(row["CT fields"])} / {int(row["NCIt synonyms"])}', readme_font) for cl, row in shared_by_cl.iterrows()],
    ('  A string that names more than one term identifies none of them: a lookup must return all candidates.', readme_font),
    ('NCIt synonyms come from the NCI EVS FLAT file, which does not say where each synonym comes from.', readme_font),
    ('', None),
    ('SHEETS', section_font),
    ('Test_Identity — one row per TESTCD + NCIt_Code scoped to the in-scope domains. Universe is', readme_font),
    ('  the wider SDTM_Test_Identity (NCI EVS), filtered to scope. Has_DSS flags', readme_font),
    ('  whether COSMoS pins the TESTCD. DSS_Count / BC_Count / DS_Codes / BC_IDs', readme_font),
    ('  are aggregated over scope DSSs only (not graph-wide).', readme_font),
    (f'Measurement_Specs — one row per DSS in {", ".join(sorted(ms_final["domain"].unique()))}. BDAGNT_value / _ncit / _codelist =', readme_font),
    ('  the binding agent the DSS assigns, its NCIt code, and the codelist it comes from.', readme_font),
    ('Axis_Meaning — the rows of Variable_Meaning (SDTM_Domain_Metadata.xlsx) that apply', readme_font),
    ('  to this file: rows for the domains in Measurement_Specs and the axis rows (scale).', readme_font),
    ('  Our position, not a published fact: Position, Basis, Basis_Note, Status as in Variable_Meaning.', readme_font),
    ('  Columns names the Measurement_Specs columns the row refers to. A variable without', readme_font),
    ('  a row: no position taken.', readme_font),
    ('  No observable key is built here (Observables Phase 2: LB, MB, MI only).', readme_font),
    ('Codelists — every codelist named in a *_codelist column of Measurement_Specs, one row', readme_font),
    ('  per term, from COSMoS_Graph_CT (SDTM CT as resolved for the graph). referenced_by =', readme_font),
    ('  the Measurement_Specs columns that name the codelist. ISBDAGT and MICROORG are complete', readme_font),
    ('  codelists, not only the terms COSMoS assigns.', readme_font),
    ('  Some terms are literally "NA" (e.g. Not Applicable in NY). Read with keep_default_na=False', readme_font),
    ('  (pandas) or the equivalent, or they become empty values.', readme_font),
    ('  To find a term, search term_submission_value, term_synonyms, term_ncit_synonyms and term_nci_preferred_term:', readme_font),
    ('  many terms are found only by a synonym, and spelling and hyphens matter.', readme_font),
    ('  Complete SDTM CT codelists from one source: a codelist in several Findings files is identical', readme_font),
    ('  in all of them (checked by Domain_Coverage).', readme_font),
    ('  term_synonyms = CDISC synonyms (SDTM CT, part of the standard). term_ncit_synonyms = NCIt', readme_font),
    ('  synonyms not already the submission value, preferred term or a CDISC synonym, compared', readme_font),
    ('  lower-cased, letters and digits only (NCI EVS FLAT file; see SOURCE VERSIONS). Separated by', readme_font),
    ('  "|", as in the NCIt FLAT file: some NCIt synonyms contain ";".', readme_font),
    ('Shared_Strings — one row per string and term, for every string (lower case, letters and digits)', readme_font),
    ('  that names more than one term within one codelist of the Codelists sheet. Source = CT (the SDTM CT', readme_font),
    ('  fields) or NCIt (term_ncit_synonyms) for that term. Term_Count = terms the string names. Shared_Via = CT fields when two or more', readme_font),
    ('  terms carry the string in their SDTM CT fields; NCIt synonyms when the sharing comes only', readme_font),
    ('  with term_ncit_synonyms. Until 2026-10-30 noon one row per string, with all terms in one cell.', readme_font),
    ('Provenance — fixed keys for programs: source versions (SDTM_CT, COSMoS, SDTMIG, SDTM, NCIt),', readme_font),
    ('  Generated, Notebook; per data sheet Rows, Columns, Content_SHA256 (equal hash = equal data).', readme_font),
    ('', None),
    ('CONSUMER-OWED DECISIONS APPLIED', section_font),
    ('Binding agent as a core column block (BDAGNT): the axis the DSSs of one BC differ by.', readme_font),
    ('LOINC: LOINC_DSS and LOINC_BC kept for shape parity with the other Findings files.', readme_font),
    (f'Allowed_Units expansion: ORRESU_codelist is expanded against CodelistTerms only when the', readme_font),
    (f'  codelist has <= {ALLOWED_UNITS_THRESHOLD} terms.', readme_font),
    ('TESTCD universe widening: yes — Test_Identity uses the full SDTM_Test_Identity', readme_font),
    ('  universe scoped to the in-scope domains, not just COSMoS-pinned TESTCDs.', readme_font),
    ('Domain class: Observation_Class joined per Measurement_Specs row.', readme_font),
    ('', None),
    ('CONFLICT FLAGS', section_font),
    ('NCIt_Code_Conflict (Test_Identity) — multiple NCIt concepts resolve from the', readme_font),
    ('  same TESTCD across the graph.', readme_font),
    ('NCIt_Reference_Disagree (Test_Identity) — NCIt at TESTCD grain disagrees', readme_font),
    ('  between graph (COSMoS pin) and reference (SDTM_Test_Identity / NCI EVS).', readme_font),
    ('Both flags come from DSS_View Test_Identity, joined on TESTCD + NCIt_Code. "Not evaluated":', readme_font),
    ('  the TESTCD + NCIt_Code is not in DSS_View (not pinned by COSMoS), or DSS_View has no', readme_font),
    ('  reference row to compare (no row in SDTM_Test_Identity).', readme_font),
    ('Rules: NCIt_Code_Conflict = Yes when COSMoS pins more than one NCIt code for the TESTCD.', readme_font),
    ('  NCIt_Reference_Disagree = Yes when the pinned NCIt code is none of the codes the reference lists', readme_font),
    ('  for the TESTCD. Lenient by design: where the reference lists more than one code for a TESTCD', readme_font),
    (f'  (today {", ".join(sorted(view_ti.loc[view_ti["Ref_NCIt_Code"].str.contains(";"), "TESTCD"])) or "none"}), "No" means the pinned code is one of them.', readme_font),
    ('', None),
    ('HEADER COLOUR CONVENTION', section_font),
    ('Green  = TESTCD / SDTM-CT-side columns (NCI EVS reference identity), and Codelists.', readme_font),
    ('Yellow = COSMoS-side columns (BC, DSS, slot pivot — pin and value_list).', readme_font),
    ('Grey   = keys, conflict flags, aggregation columns.', readme_font),
    ('Blue   = Axis_Meaning: our position (as Variable_Meaning).', readme_font),
    ('', None),
    ('STATUS', section_font),
    ('Immunogenicity sub-type — first build 2026-10. IS was earlier excluded from Specimen_Findings', readme_font),
    ('  without a file of its own; the reason for the exclusion stands.', readme_font),
    ('Not an official CDISC product.', readme_font),
]

for ri, (text, font) in enumerate(readme_lines, 1):
    cell = ws_rm.cell(row=ri, column=1, value=text if text else None)
    if font:
        cell.font = font

ws_rm.column_dimensions['A'].width = 100
print(f"ReadMe: {len(readme_lines)} lines")

ReadMe: 141 lines


In [17]:
# ── Test_Identity sheet ──
ws_ti = wb.create_sheet('Test_Identity')

TI_FILLS = {
    # Keys (grey)
    'TESTCD':                 GREY_HEADER,
    'NCIt_Code':              GREY_HEADER,
    'In_Scope_Domains':       GREY_HEADER,
    'SDTM_Domains':           GREY_HEADER,
    # Reference identity (green)
    'TEST':                   GREEN_HEADER,
    'NCIt_Preferred_Term':    GREEN_HEADER,
    'NCIt_Synonyms':          GREEN_HEADER,
    'NCIt_Definition':        GREEN_HEADER,
    'UMLS_CUI':               GREEN_HEADER,
    'NCIm_CUI':               GREEN_HEADER,
    # COSMoS coverage (yellow)
    'Has_DSS':                YELLOW_HEADER,
    'DSS_Count':              YELLOW_HEADER,
    'BC_Count':               YELLOW_HEADER,
    'DS_Codes':               YELLOW_HEADER,
    'BC_IDs':                 YELLOW_HEADER,
    # Conflict flags (grey)
    'NCIt_Code_Conflict':       GREY_HEADER,
    'NCIt_Reference_Disagree':  GREY_HEADER,
}

TI_WIDTHS = {
    'TESTCD': 14, 'NCIt_Code': 12, 'In_Scope_Domains': 18, 'SDTM_Domains': 22,
    'TEST': 38, 'NCIt_Preferred_Term': 38, 'NCIt_Synonyms': 50,
    'NCIt_Definition': 60, 'UMLS_CUI': 12, 'NCIm_CUI': 12,
    'Has_DSS': 9, 'DSS_Count': 9, 'BC_Count': 9, 'DS_Codes': 30, 'BC_IDs': 30,
    'NCIt_Code_Conflict': 12, 'NCIt_Reference_Disagree': 14,
}

write_sheet(ws_ti, ti_final, TI_FILLS, TI_WIDTHS)
print(f"Test_Identity: {len(ti_final):,} rows x {len(ti_final.columns)} cols")

Test_Identity: 100 rows x 17 cols


In [18]:
# ── Measurement_Specs sheet ──
ws_ms = wb.create_sheet('Measurement_Specs')

# Default everything to yellow (COSMoS-side); override grey for keys/flags, green for test identity.
MS_FILLS = {c: YELLOW_HEADER for c in ms_final.columns}
MS_FILLS.update({
    # Keys (grey)
    'domain': GREY_HEADER, 'Observation_Class': GREY_HEADER,
    'ds_id': GREY_HEADER, 'bc_id': GREY_HEADER, 'TESTCD': GREY_HEADER,
    # Test identity (green)
    'TEST': GREEN_HEADER, 'TESTCD_NCIt': GREEN_HEADER,
})

MS_WIDTHS = {
    'domain': 8, 'Observation_Class': 14, 'ds_id': 16, 'bc_id': 14, 'TESTCD': 14,
    'TEST': 38, 'TESTCD_NCIt': 12,
    'bc_short_name': 32, 'bc_definition': 50, 'bc_categories': 35,
    'bc_parent_label': 30, 'bc_hierarchy_path': 45, 'bc_type': 12,
    'result_scales': 18, 'bc_ncit_code': 12,
    'ds_short_name': 38, 'sdtmig_start_version': 12, 'sdtmig_end_version': 12,
    'package_date': 12, 'source': 16,
    'BDAGNT_value': 30, 'BDAGNT_ncit': 12, 'BDAGNT_codelist': 12,
    'TSTOPO_value_list': 18, 'TSTOPO_value_list_ncit': 22,
    'SPEC_value': 14, 'SPEC_ncit': 12, 'SPEC_codelist': 12,
    'METHOD_value': 22, 'METHOD_ncit': 12, 'METHOD_codelist': 14,
    'ORRESU_value': 12, 'ORRESU_ncit': 12, 'ORRESU_codelist': 14,
    'STRESU_value': 12, 'STRESU_ncit': 12, 'STRESU_codelist': 14,
    'LOINC_DSS': 14, 'LOINC_BC': 26,
    'TSTDTL_value': 22, 'TSTDTL_ncit': 12, 'TSTDTL_codelist': 14,
    'LOC_value': 18, 'LOC_ncit': 12, 'LOC_codelist': 12,
    'CAT_value': 22, 'CAT_ncit': 12, 'CAT_codelist': 14,
    'Allowed_Units': 50,
}

write_sheet(ws_ms, ms_final, MS_FILLS, MS_WIDTHS)
print(f"Measurement_Specs: {len(ms_final):,} rows x {len(ms_final.columns)} cols")

Measurement_Specs: 326 rows x 62 cols


In [19]:
# ── Axis_Meaning sheet ──
# Blue header, as Variable_Meaning in SDTM_Domain_Metadata.xlsx: our position, not a published fact.
BLUE_HEADER = PatternFill('solid', fgColor='4472C4')
ws_am = wb.create_sheet('Axis_Meaning')
AM_WIDTHS = {'Domain': 8, 'Key': 10, 'Columns': 60, 'Position': 12, 'Basis': 38, 'Basis_Note': 100, 'Status': 10}
write_sheet(ws_am, axis_meaning, {c: BLUE_HEADER for c in axis_meaning.columns}, AM_WIDTHS)
print(f"Axis_Meaning: {len(axis_meaning)} rows x {len(axis_meaning.columns)} cols")


Axis_Meaning: 6 rows x 7 cols


In [20]:
# ── Codelists sheet ── (built in the section 'Codelists and shared strings')
ws_cl = wb.create_sheet('Codelists')
CL_WIDTHS = {
    'codelist_submission_value': 14, 'codelist_concept_id': 12, 'codelist_name': 36,
    'codelist_extensible': 10, 'term_submission_value': 30, 'term_concept_id': 12,
    'term_nci_preferred_term': 36, 'term_synonyms': 40, 'term_ncit_synonyms': 40,
    'term_definition': 60, 'referenced_by': 30,
}
write_sheet(ws_cl, codelists_sheet, {c: GREEN_HEADER for c in CL_COLS}, CL_WIDTHS)
print(f"Codelists: {codelists_sheet['codelist_submission_value'].nunique()} codelists, "
      f"{len(codelists_sheet):,} terms")

# ── Shared_Strings sheet ──
ws_ss = wb.create_sheet('Shared_Strings')
SS_WIDTHS = {'codelist_submission_value': 14, 'String': 40, 'Term_Count': 10, 'Shared_Via': 14,
             'term_submission_value': 40, 'term_concept_id': 12, 'Source': 8}
write_sheet(ws_ss, shared_strings, {c: GREY_HEADER for c in shared_strings.columns}, SS_WIDTHS)
print(f"Shared_Strings: {n_shared_strings:,} strings, {len(shared_strings):,} rows")

Codelists: 7 codelists, 3,679 terms
Shared_Strings: 109 strings, 244 rows


In [21]:
wb.save(OUTPUT_FILE)
print(f"\nWritten: {OUTPUT_FILE}")
print(f"File size: {OUTPUT_FILE.stat().st_size / 1024:.0f} KB")


Written: /sessions/rcw-01ccwgzsyxjo7zktetsudi4q/mnt/cdisc-for-ai/sdtm-findings-graph/machine_actionable/Immunogenicity_Findings.xlsx
File size: 474 KB


In [22]:
# ── Provenance sheet ──
# Fixed keys for programs: the pinned source versions, and per data sheet its row count, column
# names and a content hash. The hash is computed from the file as saved, read back as text, so a
# consumer can recompute it; the xlsx file hash changes on every save, the content hash only when
# the data change.
import hashlib
from openpyxl import load_workbook as _load_workbook
from openpyxl.styles import Font as _Font, PatternFill as _PatternFill

PROVENANCE_VERSION_KEYS = ['SDTM_CT', 'COSMoS', 'SDTMIG', 'SDTM', 'NCIt']
PROVENANCE_METHOD = ('SHA-256 (UTF-8) of the sheet read as text, empty cells as "" (pandas read_excel, dtype=str): '
                     'header and rows, cells joined by tab, rows by newline. Equal hash = equal data.')


def sheet_sha256(df):
    lines = ['\t'.join(df.columns)] + ['\t'.join(row) for row in df.itertuples(index=False, name=None)]
    return hashlib.sha256('\n'.join(lines).encode('utf-8')).hexdigest()


def add_provenance(path, notebook):
    """Add a Provenance sheet (Key, Value) to a saved workbook."""
    sheets = pd.read_excel(path, sheet_name=None, dtype=str)
    if 'Provenance' in sheets:
        raise RuntimeError(f'{path.name} already has a Provenance sheet')
    rows = [(k, VERSIONS[k]) for k in PROVENANCE_VERSION_KEYS]
    rows += [('Generated', f'{datetime.now():%Y-%m-%d %H:%M}'), ('Notebook', notebook)]
    for name, df in sheets.items():
        if name == 'ReadMe':
            continue
        df = df.fillna('')
        rows += [(f'{name}.Rows', str(len(df))), (f'{name}.Columns', '|'.join(df.columns)),
                 (f'{name}.Content_SHA256', sheet_sha256(df))]
    rows.append(('Content_SHA256.Method', PROVENANCE_METHOD))
    wb_p = _load_workbook(path)
    ws_p = wb_p.create_sheet('Provenance')
    for ci, head in enumerate(['Key', 'Value'], 1):
        cell = ws_p.cell(row=1, column=ci, value=head)
        cell.font = _Font(name='Arial', bold=True, size=10, color='FFFFFF')
        cell.fill = _PatternFill('solid', fgColor='808080')
    for ri, (k, v) in enumerate(rows, 2):
        ws_p.cell(row=ri, column=1, value=k).font = _Font(name='Arial', size=10)
        ws_p.cell(row=ri, column=2, value=v).font = _Font(name='Arial', size=10)
    ws_p.column_dimensions['A'].width = 34
    ws_p.column_dimensions['B'].width = 90
    ws_p.freeze_panes = 'A2'
    wb_p.save(path)
    print(f'Provenance: {path.name}, {len(rows)} keys')

add_provenance(OUTPUT_FILE, 'sdtm-findings-graph/notebooks/Immunogenicity_Findings.ipynb')

Provenance: Immunogenicity_Findings.xlsx, 23 keys


## 6. Summary

In [23]:
print('=== Immunogenicity_Findings summary ===')
print(f'Output: {OUTPUT_FILE.relative_to(REPO_ROOT)}')
print()
print(f'Test_Identity:    {len(ti_final):>5,} rows x {len(ti_final.columns):>2} cols')
print(f'  COSMoS-pinned (Has_DSS=Yes):  {(ti_final["Has_DSS"] == "Yes").sum():>5,}')
print(f'  Coverage gap (Has_DSS=No):    {(ti_final["Has_DSS"] == "No").sum():>5,}')
print(f'  NCIt_Code_Conflict=Yes:       {(ti_final["NCIt_Code_Conflict"] == "Yes").sum():>5}')
print(f'  NCIt_Reference_Disagree=Yes:  {(ti_final["NCIt_Reference_Disagree"] == "Yes").sum():>5}')
print()
print(f'Measurement_Specs: {len(ms_final):>4,} rows x {len(ms_final.columns):>2} cols')
print(f'  Domains:                     {sorted(ms_final["domain"].unique())}')
print(f'  Slots emitted:                {slots_to_emit}')
print(f'  Allowed_Units expanded:      {(ms_final["Allowed_Units"] != "").sum()} rows')
print(f'  Binding agent codelists:     {ms_final["BDAGNT_codelist"].value_counts().to_dict()}')
print()
print(f'Axis_Meaning:     {len(axis_meaning):>5,} rows  ({", ".join(axis_meaning["Key"])})')

=== Immunogenicity_Findings summary ===
Output: sdtm-findings-graph/machine_actionable/Immunogenicity_Findings.xlsx

Test_Identity:      100 rows x 17 cols
  COSMoS-pinned (Has_DSS=Yes):      7
  Coverage gap (Has_DSS=No):       93
  NCIt_Code_Conflict=Yes:           0
  NCIt_Reference_Disagree=Yes:      0

Measurement_Specs:  326 rows x 62 cols
  Domains:                     ['IS']
  Slots emitted:                ['BDAGNT', 'SPEC', 'METHOD', 'TSTDTL', 'TSTOPO', 'ORRES', 'ORRESU', 'STRESC', 'STRESN', 'STRESU']
  Allowed_Units expanded:      0 rows
  Binding agent codelists:     {'ISBDAGT': 199, 'MICROORG': 127}

Axis_Meaning:         6 rows  (ISSPEC, ISMETHOD, ISTSTDTL, ISBDAGNT, ISTSTOPO, scale)
